### Import BAML files

In [6]:
import sys
sys.path.insert(0, '..')
from baml_client.sync_client import b
from baml_client.types import Resume

### Define baml functions

In [7]:
def example(raw_resume: str) -> Resume: 
  # BAML's internal parser guarantees ExtractResume
  # to be always return a Resume type
  response = b.ExtractResumeLlamaCppPC(raw_resume)
  return response

def example_stream(raw_resume: str) -> Resume:
  stream = b.stream.ExtractResumeLlamaCppPC(raw_resume)
  for msg in stream:
    print(msg) # This will be a PartialResume type
  
  # This will be a Resume type
  final = stream.get_final_response()
  return final

### Extract resume

In [8]:
import fitz  # PyMuPDF
import os

data_path = '../data/cv-dataset/data/INFORMATION-TECHNOLOGY'

pdf_files = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
pdf_files.sort()
pdf_files = pdf_files[:5]

all_texts = []
for pdf_file in pdf_files:
    path = os.path.join(data_path, pdf_file)
    doc = fitz.open(path)
    try:
        pdf_text = "".join(page.get_text() for page in doc)
    finally:
        doc.close()
    all_texts.append(pdf_text)

for idx, doc_text in enumerate(all_texts):
    print(f"PDF {idx+1} length: {len(doc_text)}")
    print(f"PDF {idx+1} preview: {doc_text[:100]}")

PDF 1 length: 8274
PDF 1 preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior trou
PDF 2 length: 7054
PDF 2 preview: INFORMATION TECHNOLOGY MANAGER
Professional Summary
Possesses an extensive background in Information
PDF 3 length: 5145
PDF 3 preview: WORKING RF SYSTEMS ENGINEER
Qualifications
Microsoft office/Office for Mac, pages, numbers, keynote 
PDF 4 length: 4984
PDF 4 preview: INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated IT Manager well-versed in analyzing and mitigating 
PDF 5 length: 5222
PDF 5 preview: IT MANAGEMENT
Career Overview
Detail-oriented professional with extensive Information Technology exp


### Test BAML structured output

In [9]:
# Call example() on each of the five extracted resumes, dump all outputs as JSON, one per line, into a single text file.
resumes: list[Resume] = []

output_path = "../generated/resumes_dump.jsonl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for raw_resume in all_texts:
        parsed = example(raw_resume)
        resumes.append(parsed)
        json_str = parsed.model_dump_json(indent=2)
        f.write(json_str + "\n")

print(f"All parsed resumes dumped to {output_path}")

2026-04-24T12:33:42.993 [BAML INFO] Function ExtractResumeLlamaCppPC:
    Client: LlamaCppPC (phi-4-mini-instruct-Q5_K_M.gguf) - 20340ms. StopReason: stop. Tokens(in/out): 1647/1674
    ---PROMPT---
    system: Extract from this content:
    INFORMATION TECHNOLOGY TECHNICIAN I
    Summary
    Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network security.
    Experienced in server management, systems analysis, and offering in-depth understanding of IT infrastructure areas. Detail-oriented, independent,
    and focused on taking a systematic approach to solving complex problems. Demonstrated exceptional technical knowledge and skills while
    working with various teams to achieve shared goals and objectives.
    Highlights
    Active Directory
    Group Policy Objects
    PowerShell and VBScript
    Microsoft Exchange
    VMWare experience
    New technology and product research
    Office 365 and Azure
    Stor